# Microbiome & IBD Study — Step 3: Machine Learning Models

We train and evaluate two models:
- **Black-box:** XGBoost — high performance, requires XAI for interpretation
- **White-box:** Decision Tree — lower performance but fully interpretable

**Key methodological choice:** We use `GroupKFold` for cross-validation, ensuring no patient appears in both train and validation folds — consistent with the group-based split from preprocessing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
import joblib, os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

classes  = pd.read_csv('../data/processed/class_names.csv').iloc[:, 0].tolist()
features = pd.read_csv('../data/processed/feature_names.csv').iloc[:, 0].tolist()
print('Classes:', classes)
print('Features:', len(features))

## 3.1 Select Best Split + Imbalance Strategy

We run a quick baseline Random Forest across all split × imbalance combinations using **GroupKFold** cross-validation. Best combination selected by macro F1-score.

In [ ]:
splits    = ['85_15', '80_20', '70_30']
results   = []
gkf       = GroupKFold(n_splits=5)

for split in splits:
    X_tr = np.load(f'../data/processed/X_train_{split}.npy')
    X_te = np.load(f'../data/processed/X_test_{split}.npy')
    y_tr = np.load(f'../data/processed/y_train_{split}.npy')
    y_te = np.load(f'../data/processed/y_test_{split}.npy')
    g_tr = np.load(f'../data/processed/groups_train_{split}.npy')

    # Strategy 1: class_weight (GroupKFold CV on train)
    rf_cw = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                   random_state=42, n_jobs=-1)
    rf_cw.fit(X_tr, y_tr)
    f1_cw = f1_score(y_te, rf_cw.predict(X_te), average='macro')
    results.append({'split': split.replace('_','/'), 'strategy': 'class_weight',
                    'macro_f1': round(f1_cw, 4)})

    # Strategy 2: SMOTE
    X_tr_sm = np.load(f'../data/processed/X_train_{split}_smote.npy')
    y_tr_sm = np.load(f'../data/processed/y_train_{split}_smote.npy')
    rf_sm = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_sm.fit(X_tr_sm, y_tr_sm)
    f1_sm = f1_score(y_te, rf_sm.predict(X_te), average='macro')
    results.append({'split': split.replace('_','/'), 'strategy': 'SMOTE',
                    'macro_f1': round(f1_sm, 4)})

results_df = pd.DataFrame(results).sort_values('macro_f1', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#6aab6a' if i == 0 else '#5b8db8' for i in range(len(results_df))]
bars = ax.barh(
    results_df['split'] + ' — ' + results_df['strategy'],
    results_df['macro_f1'], color=colors, edgecolor='white'
)
ax.set_xlabel('Macro F1-score')
ax.set_title('Baseline RF with GroupShuffleSplit:\nSplit × Imbalance Strategy Comparison', fontweight='bold')
ax.set_xlim(0.5, 1.0)
for bar, val in zip(bars, results_df['macro_f1']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/11_split_strategy_comparison.png', bbox_inches='tight')
plt.show()

best = results_df.iloc[0]
print(f"Best: split={best['split']}, strategy={best['strategy']}, macro F1={best['macro_f1']}")

In [ ]:
# Load best combination
best_split    = results_df.iloc[0]['split'].replace('/', '_')
best_strategy = results_df.iloc[0]['strategy']

if best_strategy == 'SMOTE':
    X_train = np.load(f'../data/processed/X_train_{best_split}_smote.npy')
    y_train = np.load(f'../data/processed/y_train_{best_split}_smote.npy')
    # groups not available for SMOTE (synthetic samples) — use None for CV
    groups_train = None
else:
    X_train      = np.load(f'../data/processed/X_train_{best_split}.npy')
    y_train      = np.load(f'../data/processed/y_train_{best_split}.npy')
    groups_train = np.load(f'../data/processed/groups_train_{best_split}.npy')

X_test  = np.load(f'../data/processed/X_test_{best_split}.npy')
y_test  = np.load(f'../data/processed/y_test_{best_split}.npy')

print(f'Using: split={best_split}, strategy={best_strategy}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

# Set up cross-validator
if groups_train is not None:
    cv = list(GroupKFold(n_splits=5).split(X_train, y_train, groups=groups_train))
    print('Using GroupKFold CV (no patient leakage)')
else:
    from sklearn.model_selection import StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    print('Using StratifiedKFold CV (SMOTE data, no groups available)')

## 3.2 Black-Box Model — XGBoost

**Why XGBoost?** It builds an ensemble of decision trees sequentially, each correcting the previous one's errors. Consistently achieves top performance on microbiome data (Su et al. 2022, Freitas et al. 2023). It is a **black-box** because individual predictions cannot be traced to input features without post-hoc XAI methods like SHAP.

In [ ]:
xgb_params = {
    'max_depth':     [3, 5, 7],
    'n_estimators':  [100, 200],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0]
}

xgb_base = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42, n_jobs=-1
)

print('Running GridSearch for XGBoost...')
xgb_grid = GridSearchCV(
    xgb_base, xgb_params, cv=cv,
    scoring='f1_macro', n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train, y_train)

print(f'\nBest XGBoost params : {xgb_grid.best_params_}')
print(f'Best CV macro F1    : {xgb_grid.best_score_:.4f}')
xgb_best = xgb_grid.best_estimator_

In [ ]:
y_pred_xgb  = xgb_best.predict(X_test)
y_proba_xgb = xgb_best.predict_proba(X_test)

print('=== XGBoost — Test Set Performance ===')
print(classification_report(y_test, y_pred_xgb, target_names=classes))
print(f'Macro F1 : {f1_score(y_test, y_pred_xgb, average="macro"):.4f}')
print(f'Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_xgb, multi_class="ovr", average="macro"):.4f}')

In [ ]:
def plot_cm(y_true, y_pred, classes, title, filepath):
    cm = confusion_matrix(y_true, y_pred).astype('float')
    cm_norm = cm / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                linewidths=0.5, ax=ax)
    ax.set_ylabel('True label', fontsize=11)
    ax.set_xlabel('Predicted label', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filepath, bbox_inches='tight')
    plt.show()

plot_cm(y_test, y_pred_xgb, classes,
        'XGBoost — Normalised Confusion Matrix',
        '../figures/12_cm_xgboost.png')

## 3.3 White-Box Model — Decision Tree

**Why Decision Tree?** Every prediction follows a path of explicit if/else rules on bacterial abundances — fully readable without any XAI tool. Manandhar et al. (2021) use Decision Tree as the interpretable baseline for IBD classification. The trade-off is typically lower performance than ensemble methods.

In [ ]:
dt_params = {
    'max_depth':        [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'criterion':        ['gini', 'entropy']
}

dt_base = DecisionTreeClassifier(class_weight='balanced', random_state=42)

print('Running GridSearch for Decision Tree...')
dt_grid = GridSearchCV(
    dt_base, dt_params, cv=cv,
    scoring='f1_macro', n_jobs=-1, verbose=1
)
dt_grid.fit(X_train, y_train)

print(f'\nBest Decision Tree params: {dt_grid.best_params_}')
print(f'Best CV macro F1         : {dt_grid.best_score_:.4f}')
dt_best = dt_grid.best_estimator_

In [ ]:
y_pred_dt  = dt_best.predict(X_test)
y_proba_dt = dt_best.predict_proba(X_test)

print('=== Decision Tree — Test Set Performance ===')
print(classification_report(y_test, y_pred_dt, target_names=classes))
print(f'Macro F1 : {f1_score(y_test, y_pred_dt, average="macro"):.4f}')
print(f'Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_dt, multi_class="ovr", average="macro"):.4f}')

In [ ]:
plot_cm(y_test, y_pred_dt, classes,
        'Decision Tree — Normalised Confusion Matrix',
        '../figures/13_cm_decision_tree.png')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_best, max_depth=3,
          feature_names=features, class_names=classes,
          filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Decision Tree — Top 3 Levels (White-Box Interpretability)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/14_decision_tree_viz.png', bbox_inches='tight', dpi=150)
plt.show()

## 3.4 Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['XGBoost (black-box)', 'Decision Tree (white-box)'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_xgb), 4),
        round(accuracy_score(y_test, y_pred_dt),  4)
    ],
    'Macro F1': [
        round(f1_score(y_test, y_pred_xgb, average='macro'), 4),
        round(f1_score(y_test, y_pred_dt,  average='macro'), 4)
    ],
    'ROC-AUC': [
        round(roc_auc_score(y_test, y_proba_xgb, multi_class='ovr', average='macro'), 4),
        round(roc_auc_score(y_test, y_proba_dt,  multi_class='ovr', average='macro'), 4)
    ]
})
print(comparison.to_string(index=False))

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
colors_cls = ['#e07b54', '#5b8db8', '#6aab6a']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model_name, y_proba) in zip(axes, [
    ('XGBoost (Black-box)', y_proba_xgb),
    ('Decision Tree (White-box)', y_proba_dt)
]):
    for i, (cls, col) in enumerate(zip(classes, colors_cls)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        ax.plot(fpr, tpr, color=col, linewidth=2,
                label=f'{cls} (AUC={auc(fpr,tpr):.3f})')
    ax.plot([0,1],[0,1],'k--', linewidth=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {model_name}', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../figures/15_roc_curves.png', bbox_inches='tight')
plt.show()

## 3.5 Critical Reflection

**Methodological improvement:** By using `GroupShuffleSplit` and `GroupKFold`, we prevent longitudinal data leakage — each patient appears entirely in either train or test, never both. This gives a realistic estimate of performance on **new, unseen patients**.

**XGBoost outperforms Decision Tree because:**
- As an ensemble of trees it captures complex non-linear interactions between bacterial species
- Gradient boosting sequentially corrects errors, reducing bias
- Consistent with Su et al. (2022) who found RF/boosting outperforms simpler models on microbiome data

**Decision Tree is still valuable because:**
- Fully interpretable without XAI — each split is a readable biological rule
- The root node species directly suggests a biological hypothesis
- Clinicians can validate each decision path

**Limitations:**
- Only 116 patients — test set covers ~17-23 patients depending on split, which is small
- Diagnosis is constant per patient (no temporal change) — we cannot model disease progression
- Results may not generalise across populations (Su et al. 2022 report AUC drops in external validation)

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(xgb_best, '../models/xgboost_best.pkl')
joblib.dump(dt_best,  '../models/decision_tree_best.pkl')
np.save('../models/X_test.npy',  X_test)
np.save('../models/y_test.npy',  y_test)
np.save('../models/X_train.npy', X_train)
np.save('../models/y_train.npy', y_train)
print('Models saved to ../models/')